# Complete Vanila RAG Pipeline

---

**Importing required packages**

In [1]:
import bs4
import requests
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader

C:\Users\aksha\AppData\Local\Temp\ipykernel_29432\3153040354.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


**Data Ingestion**

In [2]:
# Python tutorial pages for loading
urls = ["https://docs.python.org/3/tutorial/index.html",
        "https://docs.python.org/3/glossary.html#glossary",
        "https://docs.python.org/3/tutorial/appetite.html",
        "https://docs.python.org/3/tutorial/interpreter.html",
        "https://docs.python.org/3/tutorial/introduction.html", # 3
        "https://docs.python.org/3/tutorial/controlflow.html", # 4
        "https://docs.python.org/3/tutorial/datastructures.html", # 5
        "https://docs.python.org/3/tutorial/modules.html", # 6
        "https://docs.python.org/3/tutorial/inputoutput.html", # 7
        "https://docs.python.org/3/tutorial/errors.html", #8
        "https://docs.python.org/3/tutorial/classes.html",
        "https://docs.python.org/3/tutorial/stdlib.html",
        "https://docs.python.org/3/tutorial/stdlib2.html",
        "https://docs.python.org/3/tutorial/venv.html",
        "https://docs.python.org/3/tutorial/whatnow.html",
        "https://docs.python.org/3/tutorial/interactive.html",
        "https://docs.python.org/3/tutorial/floatingpoint.html",
        "https://docs.python.org/3/tutorial/appendix.html"
        ]

loader = WebBaseLoader(web_paths = urls,
                        bs_kwargs = dict(
                            parse_only = bs4.SoupStrainer(class_ = ("document"))
                        ), )
docs = loader.load()
docs

[Document(metadata={'source': 'https://docs.python.org/3/tutorial/index.html'}, page_content='\n\n\n\n\nThe Python Tutorial¶\n\nTip\nThis tutorial is designed for\nprogrammers that are new to the Python language,\nnot beginners who are new to programming.\n\nPython is an easy to learn, powerful programming language. It has efficient\nhigh-level data structures and a simple but effective approach to\nobject-oriented programming. Python’s elegant syntax and dynamic typing,\ntogether with its interpreted nature, make it an ideal language for scripting\nand rapid application development in many areas on most platforms.\nThe Python interpreter and the extensive standard library are freely available\nin source or binary form for all major platforms from the Python website,\nhttps://www.python.org/, and may be freely distributed. The same site also\ncontains distributions of and pointers to many free third party Python modules,\nprograms and tools, and additional documentation.\nThe Python in

**Chunking**

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
all_splits = text_splitter.split_documents(docs)
print("Number of chunks: ", len(all_splits))

Number of chunks:  452


**Embedding**

In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np

texts = [chunk.page_content for chunk in all_splits]
sen_trans = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model dimensions: ", sen_trans.get_embedding_dimension())

# Generating embeddings
embeddings = sen_trans.encode(texts, show_progress_bar= True, normalize_embeddings= True)
print("Embedding shape: ", embeddings.shape)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3481.14it/s]


Embedding model dimensions:  384


Batches: 100%|██████████| 15/15 [00:14<00:00,  1.01it/s]

Embedding shape:  (452, 384)


**Initializing a vectorstore**

In [7]:
# Importing required packages
import os
import numpy as np
import chromadb
from chromadb.config import Settings
import uuid

# Configurations

collection_name = "Python_Documentation"
persist_directory = "../Data/VectorStore"


# Create chromadb client and collection
os.makedirs(persist_directory, exist_ok = True)
client = chromadb.PersistentClient(path = persist_directory)

# Get or create collection
collection = client.get_or_create_collection(name = collection_name, metadata = {"Description": "HTML document embeddings for RAG"})
print("Exisiting documents in the collection: ", collection.count())

Exisiting documents in the collection:  0


**Adding records to the vectorDB**

In [8]:
# Required Args:
"""
1. Documents
2. Embeddings
"""

# Prepare data for chromadb

ids = []
metadatas = []
chunk_text = []
embeddings_list = []

for i, (chunk, embedding) in enumerate(zip(all_splits, embeddings)):
    # Generate unique id:
    doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
    ids.append(doc_id)

    # Prepare metadata
    metadata = dict(chunk.metadata)
    metadata["doc_index"] = i
    metadata["content_length"] = len(chunk.page_content)
    metadatas.append(metadata)

    # append chunk content
    chunk_text.append(chunk.page_content)

    # append the corresponding embedding
    embeddings_list.append(embedding.tolist())

# Adding the records to the chroma db
""" 
The feilds of the database are as follows:
1. ids
2. metadata
3. page_content
4. embeddings
"""

collection.add(
    ids = ids,
    embeddings = embeddings_list,
    metadatas= metadatas,
    documents = chunk_text
)

print(f"Successfully added {len(all_splits)} chunks to the vector store")
print(f"Total chunks in the collection: {collection.count()}")

Successfully added 452 chunks to the vector store
Total chunks in the collection: 452


**RAG Retriver**

In [9]:
""" 
This pipeline handles the query-based retrival from the vector-store

Required components

1. Vector Store: collection
2. Embedding generator: sen_trans

"""

# Retrieving Relevant documents for the query

query = "what is a class and how to define a class?"
top_k = 5
# score_threshold = 0.0

# generating query embedding:
query_embedding = sen_trans.encode(query, normalize_embeddings=True)


# searching the vector store for relevant records
results = collection.query(query_embeddings= [query_embedding.tolist()], n_results= top_k)

if results['documents'] and results['documents'][0]:
    documents = results['documents'][0]
    metadatas = results['metadatas'][0]
    distances = results['distances'][0]
    ids = results['ids'][0]

retrieved_docs = []

# calculating the similarity score for each retrived documents
for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
    similarity_score = 1 - distance

    retrieved_docs.append({
        'id': doc_id,
        'document' : document,
        'metadata' : metadata,
        'similarity_score': similarity_score,
        'distance' : distance,
        'rank' : i + 1
    })

print(f"Number of documents retrieved after searching {len(retrieved_docs)}")

Number of documents retrieved after searching 5


**Augmented Generation Pipeline**

In [ ]:
# Importing Reuired Libraries
import os
from dotenv import load_dotenv
from openai import OpenAI

# loading the environment variables
load_dotenv()

# get the environment variable from .env file
Llama_3_1_8B_instruct_api_key = "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"

# Create a client
llm_client = OpenAI(
    api_key= Llama_3_1_8B_instruct_api_key,
    base_url = "https://openrouter.ai/api/v1"
)

# Building the context
context = "\n\n".join([doc['document'] for doc in retrieved_docs]) if retrieved_docs else ""

# Building the prompt
prompt = f"""Use the following context to answer the question concisely.
            context : {context}
            
            query: {query}
            
            Answer: """

# Getting the response from the model
response = llm_client.chat.completions.create(
    model = "nvidia/nemotron-3-ultra-550b-a55b:free",
    messages = [{
        "role" : "user",
        "content" : prompt
    }],
    temperature= 0.1,
    max_tokens= 1024
)


In [11]:
answer = response.choices[0].message.content
print(answer)

Based on the context provided:

**What is a class?**
A class is a **template for creating user-defined objects** that bundles **data and functionality together** (attributes and methods). Creating a class creates a new type of object, allowing instances to be made with their own state.

**How to define a class?**
Use the `class` keyword followed by the class name and a colon. The body (indented) contains method definitions and class variables.
```python
class MyClass:
    """A simple example class"""
    i = 12345

    def f(self):
        return 'hello world'
```
When the definition ends, a class object is created and bound to the class name.
